In [2]:
import pandas as pd
import duckdb
from pathlib import Path

OUTPUT_PATH = Path("../data/cleaned")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f"QuickSight output folder: {OUTPUT_PATH.resolve()}")

QuickSight output folder: /Users/gerald/Documents/stb-analytics-assessment/stb-analytics-assessment/data/quicksight


In [3]:
%run "./tourism-analysis.ipynb"

(889, 6)
(73404, 8)
(210111, 9)
Occupancy raw: 889
Occupancy clean: 888
Travel mode raw: 73404
Travel mode clean: 73403
Length of stay raw: 210111
Length of stay clean: 210110
Occupancy duplicates: 0
Travel mode duplicates: 0
Length of stay duplicates: 0
/Users/gerald/Documents/stb-analytics-assessment/stb-analytics-assessment/data/raw
True


In [4]:
tourism_market_performance = connection.sql("""
    SELECT
        t.year,
        t.month,
        t.month_date,
        t.country,

        SUM(t.visitor_arrivals) AS visitor_arrivals,

        CASE
            WHEN t.country IN (
                SELECT country
                FROM average_country_rank_top10
            )
            THEN 1
            ELSE 0
        END AS is_top10

    FROM travel_mode_clean t

    WHERE t.country NOT IN ('Others', 'Not Stated')
      AND t.year NOT IN (2020, 2021, 2022)

    GROUP BY
        t.year,
        t.month,
        t.month_date,
        t.country

    ORDER BY
        t.month_date,
        t.country
""").df()

In [5]:
tourism_market_performance.head()

,year,month,month_date,country,visitor_arrivals,is_top10
0,2008,1,2008-01-01,Australia,82778.0,1
1,2008,1,2008-01-01,Bangladesh,7252.0,0
2,2008,1,2008-01-01,Belgium and Luxembourg,1574.0,0
3,2008,1,2008-01-01,Brunei Darussalam,4126.0,0
4,2008,1,2008-01-01,Canada,8760.0,0


In [6]:
tourism_market_performance.shape

(9537, 6)

In [7]:
tourism_market_performance[
    tourism_market_performance["is_top10"] == 1
]["country"].unique()

<ArrowStringArray>
[  'Australia',       'China',       'India',   'Indonesia',       'Japan',
    'Malaysia', 'Philippines', 'South Korea',          'UK',         'USA']
Length: 10, dtype: str

In [8]:
tourism_market_performance = connection.sql("""
    SELECT
        year,
        month,
        month_date,
        country,
        visitor_arrivals,
        is_top10,

        LAG(visitor_arrivals, 12) OVER (
            PARTITION BY country
            ORDER BY month_date
        ) AS previous_year_arrivals

    FROM tourism_market_performance

    ORDER BY
        country,
        month_date
""").df()


connection.register(
    "tourism_market_performance_yoy",
    tourism_market_performance
)


tourism_market_performance = connection.sql("""
    SELECT
        year,
        month,
        month_date,
        country,
        visitor_arrivals,
        is_top10,
        previous_year_arrivals,

        CASE
            WHEN previous_year_arrivals > 0
            THEN ROUND(
                (
                    visitor_arrivals - previous_year_arrivals
                ) * 100.0 / previous_year_arrivals,
                2
            )
            ELSE NULL
        END AS visitor_yoy_pct

    FROM tourism_market_performance_yoy

    ORDER BY
        country,
        month_date
""").df()

tourism_market_performance

,year,month,month_date,country,visitor_arrivals,is_top10,previous_year_arrivals,visitor_yoy_pct
0,2008,1,2008-01-01,Australia,82778.0,1,NaN,NaN
1,2008,2,2008-02-01,Australia,52457.0,1,NaN,NaN
2,2008,3,2008-03-01,Australia,61681.0,1,NaN,NaN
3,2008,4,2008-04-01,Australia,63079.0,1,NaN,NaN
4,2008,5,2008-05-01,Australia,65901.0,1,NaN,NaN
...,...,...,...,...,...,...,...,...
9532,2026,3,2026-03-01,Vietnam,22710.0,0,23240.0,-2.28
9533,2026,4,2026-04-01,Vietnam,26215.0,0,26579.0,-1.37
9534,2026,5,2026-05-01,Vietnam,24778.0,0,30069.0,-17.60
9535,2026,6,2026-06-01,Vietnam,40051.0,0,37168.0,7.76


In [9]:
tourism_market_performance["month_date"] = (
    pd.to_datetime(
        tourism_market_performance["month_date"]
    ).dt.strftime("%Y%m%d")
)

tourism_market_performance.columns.tolist()

['year',
 'month',
 'month_date',
 'country',
 'visitor_arrivals',
 'is_top10',
 'previous_year_arrivals',
 'visitor_yoy_pct']

In [10]:
top10_demographics = connection.sql("""
    WITH demographic_totals AS (
        SELECT
            year,
            country,
            age_group,
            sex,
            SUM(visitor_arrivals) AS demographic_visitors

        FROM length_of_stay_clean

        WHERE year NOT IN (2020, 2021, 2022)
          AND country IN (
              SELECT country
              FROM average_country_rank_top10
          )
          AND country NOT IN ('Others', 'Not Stated')
          AND age_group <> 'Not Stated'
          AND sex <> 'Not Stated'

        GROUP BY
            year,
            country,
            age_group,
            sex
    ),

    market_totals AS (
        SELECT
            year,
            country,
            SUM(demographic_visitors) AS total_market_visitors

        FROM demographic_totals

        GROUP BY
            year,
            country
    )

    SELECT
        d.year,
        d.country,
        d.age_group,
        d.sex,
        d.demographic_visitors,
        m.total_market_visitors,

        ROUND(
            d.demographic_visitors * 100.0
            / NULLIF(m.total_market_visitors, 0),
            2
        ) AS demographic_share_pct

    FROM demographic_totals d

    LEFT JOIN market_totals m
        ON d.year = m.year
        AND d.country = m.country

    ORDER BY
        d.year,
        d.country,
        demographic_share_pct DESC
""").df()

top10_demographics

,year,country,age_group,sex,demographic_visitors,total_market_visitors,demographic_share_pct
0,2008,Australia,Age 45 - 54,Male,94972.0,787373.0,12.06
1,2008,Australia,Age 35 - 44,Male,86026.0,787373.0,10.93
2,2008,Australia,Age 55 - 64,Male,79202.0,787373.0,10.06
3,2008,Australia,Age 25 - 34,Male,73264.0,787373.0,9.30
4,2008,Australia,Age 45 - 54,Female,65046.0,787373.0,8.26
...,...,...,...,...,...,...,...
2555,2026,USA,Age 14 & Below,Female,16308.0,443435.0,3.68
2556,2026,USA,Age 20 - 24,Male,12282.0,443435.0,2.77
2557,2026,USA,Age 20 - 24,Female,11876.0,443435.0,2.68
2558,2026,USA,Age 15 - 19,Female,7547.0,443435.0,1.70


In [11]:
top10_demographics = connection.sql("""
    SELECT
        d.*,
        r.avg_rank,
        r.best_rank,
        r.worst_rank,
        r.years_observed

    FROM top10_demographics d

    LEFT JOIN avg_country_demographic_rank_top10 r
        ON d.country = r.country
        AND d.age_group = r.age_group
        AND d.sex = r.sex

    ORDER BY
        d.country,
        d.year,
        d.demographic_share_pct DESC
""").df()

top10_demographics

,year,country,age_group,sex,demographic_visitors,total_market_visitors,demographic_share_pct,avg_rank,best_rank,worst_rank,years_observed
0,2008,Australia,Age 45 - 54,Male,94972.0,787373.0,12.06,1.31,1,2,16
1,2008,Australia,Age 35 - 44,Male,86026.0,787373.0,10.93,1.69,1,2,16
2,2008,Australia,Age 55 - 64,Male,79202.0,787373.0,10.06,4.44,3,9,16
3,2008,Australia,Age 25 - 34,Male,73264.0,787373.0,9.30,5.13,3,9,16
4,2008,Australia,Age 45 - 54,Female,65046.0,787373.0,8.26,6.31,4,8,16
...,...,...,...,...,...,...,...,...,...,...,...
2555,2026,USA,Age 14 & Below,Female,16308.0,443435.0,3.68,12.00,12,12,16
2556,2026,USA,Age 20 - 24,Male,12282.0,443435.0,2.77,13.50,13,14,16
2557,2026,USA,Age 20 - 24,Female,11876.0,443435.0,2.68,13.50,13,14,16
2558,2026,USA,Age 15 - 19,Female,7547.0,443435.0,1.70,15.19,15,16,16


In [12]:
concert_monthly_analysis = country_demographic_concert.copy()

concert_monthly_analysis["month_date"] = pd.to_datetime(
    dict(
        year=concert_monthly_analysis["year"],
        month=concert_monthly_analysis["month"],
        day=1
    )
)

concert_monthly_analysis.head()

,year,month,country,age_group,sex,visitor_yoy_pct,concert_count,has_concert,month_date
0,2009,1,Australia,Age 14 & Below,Female,3.53,0,0,2009-01-01
1,2009,2,Australia,Age 14 & Below,Female,-9.08,0,0,2009-02-01
2,2009,3,Australia,Age 14 & Below,Female,-24.07,1,1,2009-03-01
3,2009,4,Australia,Age 14 & Below,Female,24.42,1,1,2009-04-01
4,2009,5,Australia,Age 14 & Below,Female,0.77,0,0,2009-05-01


In [13]:
concert_monthly_analysis["concert_status"] = (
    concert_monthly_analysis["has_concert"]
    .map({
        1: "Concert Month",
        0: "No Concert"
    })
)

In [14]:
concert_monthly_analysis["target_demographic"] = (
    (
        (concert_monthly_analysis["sex"] == "Male")
        &
        (concert_monthly_analysis["age_group"].isin(
            ["25 - 34", "35 - 44", "45 - 54"]
        ))
    )
    |
    (
        (concert_monthly_analysis["sex"] == "Female")
        &
        (concert_monthly_analysis["age_group"].isin(
            ["25 - 34", "35 - 44"]
        ))
    )
).map({
    True: "Key Demographic",
    False: "Other Demographic"
})

# Unique identifier for each country-demographic-month observation in Looker Studio
concert_monthly_analysis["observation"] = (
    concert_monthly_analysis["country"].astype(str)
    + " | "
    + concert_monthly_analysis["age_group"].astype(str)
    + " | "
    + concert_monthly_analysis["sex"].astype(str)
    + " | "
    + concert_monthly_analysis["month_date"].astype(str)
)


In [15]:
concert_monthly_analysis["age_group"].unique()

<ArrowStringArray>
['Age 14 & Below',    'Age 15 - 19',    'Age 20 - 24',    'Age 25 - 34',
    'Age 35 - 44',    'Age 45 - 54',    'Age 55 - 64', 'Age 65 & Above']
Length: 8, dtype: str

In [16]:
concert_statistical_results = connection.sql("""
    SELECT
        b.country,
        b.age_group,
        b.sex,

        b.concert_months,
        b.no_concert_months,

        ROUND(b.concert_avg_yoy, 2)
            AS concert_avg_yoy,

        ROUND(b.no_concert_avg_yoy, 2)
            AS no_concert_avg_yoy,

        ROUND(b.uplift_pct_point, 2)
            AS uplift_pct_point,

        ROUND(b.t_statistic, 3)
            AS t_statistic,

        ROUND(b.p_value, 4)
            AS binary_p_value,

        ROUND(i.correlation, 3)
            AS correlation,

        ROUND(i.p_value, 4)
            AS intensity_p_value,

        i.observations,

        i.result
            AS intensity_result

    FROM binary_test_results b

    LEFT JOIN final_intensity_concert_analysis i
        ON b.country = i.country
        AND b.age_group = i.age_group
        AND b.sex = i.sex

    ORDER BY
        b.country,
        b.age_group,
        b.sex
""").df()

concert_statistical_results

,country,age_group,sex,concert_months,no_concert_months,concert_avg_yoy,no_concert_avg_yoy,uplift_pct_point,t_statistic,binary_p_value,correlation,intensity_p_value,observations,intensity_result
0,Australia,Age 14 & Below,Female,125,38,6.66,9.05,-2.39,-0.824,0.4132,-0.025,0.7492,163,Not Significant
1,Australia,Age 14 & Below,Male,125,38,6.83,9.35,-2.52,-0.945,0.3484,-0.006,0.9393,163,Not Significant
2,Australia,Age 15 - 19,Female,125,38,5.54,4.35,1.19,0.338,0.7367,0.109,0.1679,163,Not Significant
3,Australia,Age 15 - 19,Male,125,38,5.69,2.30,3.39,1.146,0.2553,0.180,0.0212,163,Significant Positive
4,Australia,Age 20 - 24,Female,125,38,3.97,-1.83,5.79,2.256,0.0276,0.228,0.0035,163,Significant Positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,Vietnam,Age 45 - 54,Male,125,38,1.82,8.98,-7.15,-2.484,0.0163,-0.283,0.0003,163,Significant Negative
806,Vietnam,Age 55 - 64,Female,125,38,8.11,22.38,-14.27,-2.931,0.0052,-0.352,0.0000,163,Significant Negative
807,Vietnam,Age 55 - 64,Male,125,38,4.86,15.82,-10.97,-3.401,0.0012,-0.358,0.0000,163,Significant Negative
808,Vietnam,Age 65 & Above,Female,125,38,8.94,18.14,-9.20,-2.001,0.0507,-0.236,0.0024,163,Significant Negative


In [17]:
concert_statistical_results["binary_result"] = (
    concert_statistical_results.apply(
        lambda row:
            "Significant Positive"
            if (
                row["binary_p_value"] < 0.05
                and row["uplift_pct_point"] > 0
            )
            else (
                "Significant Negative"
                if (
                    row["binary_p_value"] < 0.05
                    and row["uplift_pct_point"] < 0
                )
                else "Not Significant"
            ),
        axis=1
    )
)

In [18]:
top10_markets = set(
    average_country_rank_top10["country"]
)

concert_statistical_results["is_top10"] = (
    concert_statistical_results["country"]
    .isin(top10_markets)
    .astype(int)
)

In [19]:
binary_top10_count = (
    concert_statistical_results[
        (concert_statistical_results["is_top10"] == 1)
        &
        (
            concert_statistical_results["binary_result"]
            == "Significant Positive"
        )
    ]["country"]
    .nunique()
)

print(
    "Top 10 markets with significant positive "
    f"binary result: {binary_top10_count}/10"
)

Top 10 markets with significant positive binary result: 6/10


In [20]:
intensity_top10_count = (
    concert_statistical_results[
        (concert_statistical_results["is_top10"] == 1)
        &
        (
            concert_statistical_results["intensity_result"]
            == "Significant Positive"
        )
    ]["country"]
    .nunique()
)

print(
    "Top 10 markets with significant positive "
    f"concert-intensity relationship: {intensity_top10_count}/10"
)

Top 10 markets with significant positive concert-intensity relationship: 7/10


In [21]:
tables = {
    "tourism_market_performance":
        tourism_market_performance,

    "top10_demographics":
        top10_demographics,

    "concert_monthly_analysis":
        concert_monthly_analysis,

    "concert_statistical_results":
        concert_statistical_results
}

for name, df in tables.items():

    print("=" * 60)
    print(name)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("\nNull values:")
    print(
        df.isnull()
        .sum()
        .loc[lambda x: x > 0]
    )

    print()

tourism_market_performance
Rows: 9537
Columns: 8

Null values:
previous_year_arrivals    612
visitor_yoy_pct           612
dtype: int64

top10_demographics
Rows: 2560
Columns: 11

Null values:
Series([], dtype: int64)

concert_monthly_analysis
Rows: 132357
Columns: 12

Null values:
visitor_yoy_pct    130
dtype: int64

concert_statistical_results
Rows: 810
Columns: 16

Null values:
Series([], dtype: int64)



In [22]:
for name, df in tables.items():
    duplicates = df.duplicated().sum()
    print(
        f"{name}: {duplicates} duplicate rows"
    )

tourism_market_performance: 0 duplicate rows
top10_demographics: 0 duplicate rows
concert_monthly_analysis: 0 duplicate rows
concert_statistical_results: 0 duplicate rows


In [23]:
tourism_market_performance.to_csv(
    OUTPUT_PATH / "tourism_market_performance.csv",
    index=False
)

top10_demographics.to_csv(
    OUTPUT_PATH / "top10_demographics.csv",
    index=False
)

concert_monthly_analysis.to_csv(
    OUTPUT_PATH / "concert_monthly_analysis.csv",
    index=False
)

concert_statistical_results.to_csv(
    OUTPUT_PATH / "concert_statistical_results.csv",
    index=False
)

In [24]:
for file in OUTPUT_PATH.glob("*.csv"):

    size_mb = file.stat().st_size / (1024 ** 2)

    print(
        f"{file.name}: "
        f"{size_mb:.2f} MB"
    )

top10_demographics.csv: 0.16 MB
tourism_market_performance.csv: 0.46 MB
concert_statistical_results.csv: 0.09 MB
concert_monthly_analysis.csv: 17.66 MB


In [25]:
summary = pd.DataFrame({
    "dataset": [
        "tourism_market_performance",
        "top10_demographics",
        "concert_monthly_analysis",
        "concert_statistical_results"
    ],

    "rows": [
        len(tourism_market_performance),
        len(top10_demographics),
        len(concert_monthly_analysis),
        len(concert_statistical_results)
    ],

    "columns": [
        len(tourism_market_performance.columns),
        len(top10_demographics.columns),
        len(concert_monthly_analysis.columns),
        len(concert_statistical_results.columns)
    ]
})

summary

,dataset,rows,columns
0,tourism_market_performance,9537,8
1,top10_demographics,2560,11
2,concert_monthly_analysis,132357,12
3,concert_statistical_results,810,16


In [26]:
print(
    concert_monthly_analysis["concert_count"]
    .value_counts()
    .sort_index()
)

print(
    "Minimum:",
    concert_monthly_analysis["concert_count"].min()
)

print(
    "Maximum:",
    concert_monthly_analysis["concert_count"].max()
)

print(
    "Unique values:",
    sorted(
        concert_monthly_analysis["concert_count"]
        .dropna()
        .unique()
    )
)

concert_count
0    30871
1    34908
2    29214
3    17860
4    12186
5     3253
6     3252
7      813
Name: count, dtype: int64
Minimum: 0
Maximum: 7
Unique values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
